In [ ]:
!pip -q install monai zarr numcodecs

In [ ]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import json
import glob
import time
import random

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import zarr

from scipy.ndimage import gaussian_filter

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.model_selection import GroupKFold

torch.backends.cudnn.benchmark = True

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Configuration
# ============================================================

DATA_DIR = Path(

    "/kaggle/input/competitions/biohub-cell-tracking-during-development"

)

TRAIN_DIR = DATA_DIR / "train"

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(DEVICE)

In [ ]:
# ============================================================
# Training Configuration
# ============================================================

POOL = 2

PATCH_Z = 64

PATCH_Y = 128

PATCH_X = 128

SIGMA_Z = 1.0

SIGMA_XY = 2.5

BATCH_SIZE = 4

NUM_WORKERS = 4

EPOCHS = 10

LR = 1e-3

WEIGHT_DECAY = 1e-4

AMP = True

In [ ]:
# ============================================================
# Dataset
# ============================================================

movies = sorted(

    TRAIN_DIR.glob("*.zarr")

)

print(

    "Movies :", len(movies)

)

In [ ]:
# ============================================================
# Read Ground Truth
# ============================================================

def read_nodes(geff):

    gt = zarr.open(

        geff,

        mode="r"

    )

    return pd.DataFrame({

        "t":

            np.array(

                gt["nodes"]["props"]["t"]["values"]

            ),

        "z":

            np.array(

                gt["nodes"]["props"]["z"]["values"]

            ),

        "y":

            np.array(

                gt["nodes"]["props"]["y"]["values"]

            ),

        "x":

            np.array(

                gt["nodes"]["props"]["x"]["values"]

            )

    })

In [ ]:
# ============================================================
# Better Normalization
# ============================================================

def normalize(img):

    img = img.astype(np.float32)

    p1 = np.percentile(

        img,

        1

    )

    p998 = np.percentile(

        img,

        99.8

    )

    img = np.clip(

        img,

        p1,

        p998

    )

    img = (

        img-p1

    )/(

        p998-p1+1e-8

    )

    return img

In [ ]:
def pool_xy(vol):

    return vol[

        :,

        ::POOL,

        ::POOL

    ]

In [ ]:
stats=[]

for movie in movies:

    gt=read_nodes(

        movie.with_suffix(".geff")

    )

    stats.append({

        "movie":movie.stem,

        "cells":len(gt),

        "frames":gt.t.nunique()

    })

stats=pd.DataFrame(stats)

display(stats.head())

In [ ]:
# ============================================================
# Build Movie Statistics
# ============================================================

movie_stats = {}

for movie in tqdm(movies, desc="Movie Statistics"):

    volume = zarr.open(movie, mode="r")["0"]

    n = volume.shape[0]

    sample_idx = np.linspace(
        0,
        n - 1,
        min(8, n),
        dtype=int
    )

    pixels = []

    for i in sample_idx:

        pixels.append(volume[i].ravel())

    pixels = np.concatenate(pixels)

    movie_stats[str(movie)] = {

        "p1": np.percentile(pixels, 1),

        "p998": np.percentile(pixels, 99.8)

    }

print("Statistics created.")

In [ ]:
# ============================================================
# Fast Heatmap
# ============================================================

def create_heatmap(nodes):

    heat = np.zeros(

        (

            PATCH_Z,

            PATCH_Y,

            PATCH_X

        ),

        dtype=np.float32

    )

    if len(nodes)==0:

        return heat

    coords = nodes[

        ["z","y","x"]

    ].to_numpy(np.int32)

    z = coords[:,0]

    y = coords[:,1]//POOL

    x = coords[:,2]//POOL

    valid = (

        (z>=0)&

        (z<PATCH_Z)&

        (y>=0)&

        (y<PATCH_Y)&

        (x>=0)&

        (x<PATCH_X)

    )

    heat[

        z[valid],

        y[valid],

        x[valid]

    ]=1

    gaussian_filter(

        heat,

        sigma=(

            SIGMA_Z,

            SIGMA_XY,

            SIGMA_XY

        ),

        output=heat

    )

    m = heat.max()

    if m>0:

        heat/=m

    return heat

In [ ]:
# ============================================================
# Precompute Gaussian Kernel
# ============================================================

KZ = 9
KY = 17
KX = 17

SIGMA_Z = 1.0
SIGMA_XY = 2.5

zz, yy, xx = np.meshgrid(

    np.arange(KZ),

    np.arange(KY),

    np.arange(KX),

    indexing="ij"

)

cz = (KZ - 1) / 2
cy = (KY - 1) / 2
cx = (KX - 1) / 2

GAUSSIAN_KERNEL = np.exp(

    -(

        ((zz-cz)**2)/(2*SIGMA_Z**2)

        +

        ((yy-cy)**2)/(2*SIGMA_XY**2)

        +

        ((xx-cx)**2)/(2*SIGMA_XY**2)

    )

).astype(np.float32)

GAUSSIAN_KERNEL /= GAUSSIAN_KERNEL.max()

print(GAUSSIAN_KERNEL.shape)

In [ ]:
# ============================================================
# Build Frame Index
# ============================================================

frame_index = []

for movie in tqdm(movies):

    nodes = read_nodes(

        movie.with_suffix(".geff")

    )

    grouped = {

        t:g.reset_index(drop=True)

        for t,g in nodes.groupby("t")

    }

    volume = zarr.open(

        movie,

        mode="r"

    )["0"]

    for t in range(volume.shape[0]):

        frame_index.append({

            "movie":str(movie),

            "time":t,

            "nodes":grouped.get(

                t,

                pd.DataFrame(

                    columns=["z","y","x"]

                )

            )

        })

frame_index = pd.DataFrame(

    frame_index

)

print(frame_index.shape)

In [ ]:
frame_index["embryo"] = frame_index.movie.apply(

    lambda x: Path(x).stem

)

gkf = GroupKFold(

    n_splits=5

)

train_idx,val_idx = next(

    gkf.split(

        frame_index,

        groups=frame_index.embryo

    )

)

train_df = frame_index.iloc[train_idx].reset_index(drop=True)

val_df = frame_index.iloc[val_idx].reset_index(drop=True)

print(

    len(train_df),

    len(val_df)

)

In [ ]:
# ============================================================
# Dataset
# ============================================================

class CellDataset(Dataset):

    def __init__(

        self,

        dataframe,

        training=True

    ):

        self.df = dataframe.reset_index(drop=True)

        self.training = training

        self.movies = {}

        for movie in self.df.movie.unique():

            self.movies[movie] = zarr.open(

                movie,

                mode="r"

            )["0"]

    def make_heatmap(self, nodes):

        heat = np.zeros(

            (

                PATCH_Z,

                PATCH_Y,

                PATCH_X

            ),

            dtype=np.float32

        )

        if len(nodes)==0:

            return heat

        kz,ky,kx = GAUSSIAN_KERNEL.shape

        rz = kz//2
        ry = ky//2
        rx = kx//2

        coords = nodes[

            ["z","y","x"]

        ].to_numpy(

            np.int32

        )

        for z,y,x in coords:

            y//=POOL
            x//=POOL

            z1=max(0,z-rz)
            y1=max(0,y-ry)
            x1=max(0,x-rx)

            z2=min(PATCH_Z,z+rz+1)
            y2=min(PATCH_Y,y+ry+1)
            x2=min(PATCH_X,x+rx+1)

            kz1=rz-(z-z1)
            ky1=ry-(y-y1)
            kx1=rx-(x-x1)

            kz2=kz1+(z2-z1)
            ky2=ky1+(y2-y1)
            kx2=kx1+(x2-x1)

            heat[

                z1:z2,

                y1:y2,

                x1:x2

            ] = np.maximum(

                heat[

                    z1:z2,

                    y1:y2,

                    x1:x2

                ],

                GAUSSIAN_KERNEL[

                    kz1:kz2,

                    ky1:ky2,

                    kx1:kx2

                ]

            )

        return heat

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        volume = self.movies[row.movie]

        frame = volume[row.time].astype(np.float32)

        stat = movie_stats[row.movie]

        frame = np.clip(

            frame,

            stat["p1"],

            stat["p998"]

        )

        frame = (

            frame-stat["p1"]

        )/(

            stat["p998"]-stat["p1"]+1e-8

        )

        frame = frame[:,::POOL,::POOL]

        heat = self.make_heatmap(

            row.nodes

        )

        if self.training:

            if random.random()<0.5:

                frame = frame[:,:,::-1].copy()

                heat = heat[:,:,::-1].copy()

            if random.random()<0.5:

                frame = frame[:,::-1,:].copy()

                heat = heat[:,::-1,:].copy()

        frame = torch.from_numpy(

            frame

        ).unsqueeze(0)

        heat = torch.from_numpy(

            heat

        ).unsqueeze(0)

        return frame.float(), heat.float()

In [ ]:
train_dataset = CellDataset(train_df, True)
val_dataset = CellDataset(val_df, False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=False,
    drop_last=True
)

In [ ]:
images,targets = next(

    iter(train_loader)

)

print(images.shape)

print(targets.shape)

print(images.min(),images.max())

print(targets.min(),targets.max())

In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceLoss

from torch.amp import GradScaler
from torch.amp import autocast

In [ ]:
# ============================================================
# Model
# ============================================================

model = UNet(

    spatial_dims=3,

    in_channels=1,

    out_channels=1,

    channels=(32,64,128,256),

    strides=(2,2,2),

    num_res_units=2,

    dropout=0.10,

    norm="INSTANCE",

).to(DEVICE)

print(model)

In [ ]:
total_params = sum(

    p.numel()

    for p in model.parameters()

)

trainable = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print(f"Parameters : {total_params:,}")

print(f"Trainable : {trainable:,}")

In [ ]:
# ============================================================
# Binary Focal Loss
# ============================================================

class BinaryFocalLoss(nn.Module):

    def __init__(

        self,

        gamma=2,

        alpha=0.25

    ):

        super().__init__()

        self.gamma = gamma

        self.alpha = alpha

    def forward(

        self,

        logits,

        targets

    ):

        bce = F.binary_cross_entropy_with_logits(

            logits,

            targets,

            reduction="none"

        )

        prob = torch.sigmoid(logits)

        pt = torch.where(

            targets == 1,

            prob,

            1 - prob

        )

        loss = (

            self.alpha

            *

            (1 - pt).pow(self.gamma)

            *

            bce

        )

        return loss.mean()

In [ ]:
# ============================================================
# Hybrid Loss
# ============================================================

dice_loss = DiceLoss(

    sigmoid=True

)

bce_loss = nn.BCEWithLogitsLoss()

focal_loss = BinaryFocalLoss()


def criterion(

    pred,

    target

):

    d = dice_loss(

        pred,

        target

    )

    b = bce_loss(

        pred,

        target

    )

    f = focal_loss(

        pred,

        target

    )

    return (

        0.4 * b

        +

        0.4 * d

        +

        0.2 * f

    )

In [ ]:
PATIENCE = 5

# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-3,

    weight_decay=1e-4

)

In [ ]:
# ============================================================
# OneCycle Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.OneCycleLR(

    optimizer,

    max_lr=1e-3,

    epochs=EPOCHS,

    steps_per_epoch=len(train_loader),

    pct_start=0.1,

    anneal_strategy="cos",

    div_factor=25,

    final_div_factor=1000

)

In [ ]:
scaler = GradScaler(
    "cuda",
    enabled=AMP
)

In [ ]:
class ModelEMA:

    def __init__(

        self,

        model,

        decay=0.999

    ):

        self.decay = decay

        self.shadow = {}

        for name, param in model.named_parameters():

            if param.requires_grad:

                self.shadow[name] = param.data.clone()

    def update(

        self,

        model

    ):

        with torch.no_grad():

            for name, param in model.named_parameters():

                if not param.requires_grad:

                    continue

                self.shadow[name].mul_(

                    self.decay

                ).add_(

                    param.data,

                    alpha=1-self.decay

                )

In [ ]:
ema = ModelEMA(model)

In [ ]:
images, targets = next(

    iter(train_loader)

)

images = images.to(DEVICE)

with torch.no_grad():

    outputs = model(images)

print("Input :", images.shape)

print("Output:", outputs.shape)

In [ ]:
# ============================================================
# Training State
# ============================================================

best_loss = float("inf")

patience = 5

history = {

    "train": [],

    "val": []

}

In [ ]:
# ============================================================
# Train One Epoch
# ============================================================

from tqdm.auto import tqdm

def train_one_epoch():

    model.train()

    running_loss = 0.0

    progress = tqdm(train_loader, leave=False)

    for images, targets in progress:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last_3d
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type="cuda",
            enabled=AMP
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                targets
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)

        scaler.update()

        scheduler.step()

        ema.update(model)

        running_loss += loss.item()

        progress.set_postfix(

            loss=f"{loss.item():.4f}",

            lr=f"{scheduler.get_last_lr()[0]:.2e}"

        )

    return running_loss / len(train_loader)

In [ ]:
# ============================================================
# Validation
# ============================================================

@torch.no_grad()

def validate():

    model.eval()

    running_loss = 0.0

    progress = tqdm(val_loader, leave=False)

    for images, targets in progress:

        images = images.to(
            DEVICE,
            non_blocking=True,
            memory_format=torch.channels_last_3d
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True
        )

        with autocast(
            device_type="cuda",
            enabled=AMP
        ):

            outputs = model(images)

            loss = criterion(
                outputs,
                targets
            )

        running_loss += loss.item()

        progress.set_postfix(

            loss=f"{loss.item():.4f}"

        )

    return running_loss / len(val_loader)

In [ ]:
# ============================================================
# Training Loop
# ============================================================

best_loss = float("inf")

patience = 0

MIN_DELTA = 1e-3

history = {

    "train": [],

    "val": []

}

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_one_epoch()

    val_loss = validate()

    history["train"].append(train_loss)

    history["val"].append(val_loss)

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Valid Loss : {val_loss:.4f}")

    if val_loss < best_loss - MIN_DELTA:

        best_loss = val_loss

        patience = 0

        checkpoint = {

            "epoch": epoch + 1,

            "model_state_dict": model.state_dict(),

            "optimizer_state_dict": optimizer.state_dict(),

            "scheduler_state_dict": scheduler.state_dict(),

            "scaler_state_dict": scaler.state_dict(),

            "best_loss": best_loss,

            "history": history

        }

        torch.save(

            checkpoint,

            "best_checkpoint.pth"

        )

        torch.save(

            model.state_dict(),

            "best_model_weights.pth"

        )

        print("✓ Best model saved.")

    else:

        patience += 1

        print(f"No improvement ({patience}/{PATIENCE})")

    torch.save(

        checkpoint,

        "last_checkpoint.pth"

    )

    if patience >= PATIENCE:

        print("Early stopping.")

        break

In [ ]:
import os

print("=" * 50)
print("Training Finished")
print("=" * 50)

for file in os.listdir("."):
    if file.endswith(".pth"):
        size = os.path.getsize(file) / 1024**2
        print(f"{file:25s} {size:.2f} MB")

In [ ]:
# ============================================================
# Save Best Model
# ============================================================

MIN_DELTA = 1e-3

if val_loss < best_loss - MIN_DELTA:

    best_loss = val_loss
    patience = 0

    checkpoint = {

        "epoch": epoch + 1,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "scaler_state_dict": scaler.state_dict(),

        "best_loss": best_loss,

        "train_loss": train_loss,

        "val_loss": val_loss,

        "history": history,

    }

    # Full checkpoint (resume training)
    torch.save(
        checkpoint,
        "best_checkpoint.pth"
    )

    # Weights only (inference)
    torch.save(
        model.state_dict(),
        "best_model_weights.pth"
    )

    print(f"✓ Best model saved (Val Loss: {best_loss:.4f})")

else:

    patience += 1

    print(f"No improvement ({patience}/{PATIENCE})")

# Save latest checkpoint every epoch
last_checkpoint = {

    "epoch": epoch + 1,

    "model_state_dict": model.state_dict(),

    "optimizer_state_dict": optimizer.state_dict(),

    "scheduler_state_dict": scheduler.state_dict(),

    "scaler_state_dict": scaler.state_dict(),

    "best_loss": best_loss,

    "train_loss": train_loss,

    "val_loss": val_loss,

    "history": history,

}

torch.save(
    last_checkpoint,
    "last_checkpoint.pth"
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(

    history["train"],

    label="Train"

)

plt.plot(

    history["val"],

    label="Validation"

)

plt.legend()

plt.grid(True)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.show()